# **Goal**

The purpose of this notebook is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

###**Pythia Model Family**

- Pythia models were originally designed for interpretability research.

- All Pythia models were trained on the same data in the same order.

- Training data: All Pythia models are trained on The Pile. The Pile is a 825GiB general-purpose dataset in English. It contains texts from 22 diverse sources, roughly broken down into five categories: academic writing (e.g. arXiv), internet (e.g. CommonCrawl), prose (e.g. Project Gutenberg), dialogue (e.g. YouTube subtitles), and miscellaneous (e.g. GitHub, Enron Emails).

- Each model saw 299,892,736,000 ~= 300B tokens during training.

####**Pythia 1B**

- <u>Total params</u>: 1,011,781,632. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 805,736,448. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 16
- <u>Model dimension</u>: 2048. I.e.: Every token is represented internally by a vector of length 2048.
- <u>MLP units per layer:</u> 8192
- <u>Heads</u>: 8
- <u>Vocabulary size:</u> 50,304. Amount of tokens in vocabulary.
- <u>Max context window</u>: 2048 tokens.

Load model

In [1]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

model = GPTNeoXForCausalLM.from_pretrained(
  "EleutherAI/pythia-1b-deduped",
  revision="step143000",
  cache_dir="./pythia-c-deduped/step143000",
)

tokenizer = AutoTokenizer.from_pretrained(
  "EleutherAI/pythia-1b-deduped",
  revision="step143000",
  cache_dir="./pythia-1b-deduped/step143000",
)

inputs = tokenizer("Hello, I am", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.02GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/4.64k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=24) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'Hello, I am a newbie to the world of web development. I am trying to create a simple website using HTML'

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [2]:
inputs = tokenizer("How much do you like ice cream?", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=28) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'How much do you like ice cream?\n\nI like it all. I like it all. I like it all. I like it'

In [3]:
inputs = tokenizer("The capital of France is", return_tensors="pt")
tokens = model.generate(**inputs)
tokenizer.decode(tokens[0])

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=25) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'The capital of France is Paris.\n\nThe capital of France is Paris.\n\nThe capital of France is Paris.'

In [4]:

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(output[0]))

The capital of France is the place to be if you're looking for a cultural experience like no other.

The Arc de Triomphe, the iconic monument to the imperial past and the future of Paris, has been open for more than 150 years. With its two


In [5]:
inputs = tokenizer("What is love?", return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Input tokens: tensor([1276,  310, 2389,   32])
Output tokens: tensor([  187,  2513,   352,   247,  5471,    13,   390,  1057,   352,  1599,
         1633,  2010,   865,   187,     3,  4374,   449,   754,  9577,  4541,
           15,   346,  1147,  2097, 18248,  3095,   665,  5605,   281,   320,
         7636,   407,   368,   347,  1199,   347,   309,   513,    15,   733,
          434,   417,   594,   326,  4130,   476,   755,   619, 21909,  4354])

What is love?
Is it a feeling, or does it mean something else?"
"Yes." He answered quickly. "It means loving someone who wants to be loved by you as much as I do. It's not so that everyone can get my affection easily


Here we are just printing the internal shape of one layer of the model.

In [6]:
print(model.gpt_neox.layers[0].mlp)

GPTNeoXMLP(
  (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
  (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
  (act): GELUActivation()
)


In [7]:
model.config.intermediate_size

8192

The input vector representation is 2048-dimensional. The number of MLP units in the layer is 8192. Each of these produces a scalar from the input through learned weights and bises, so the layer produces a transformed and expanded 8192-d vector. After activation using GELU (Gaussian Error Linear Unit), these values are transformed back into a new 2048-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{GELU}(x) = x \cdot \Phi(x) = 0.5x \left(1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right)$, where $Φ(x)$ is the cumulative distribution function of the standard normal distribution, and $\text{erf}(x)$ is the Gauss error function.

**MLP Unit Ablation Experiments**

In [8]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # Load weights in layer layer_idx
    mlp = model.gpt_neox.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    # Indexes in [0, 8191]
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    #
    def ablation_hook(module, inputs, output):
        indices_device = indices.to(output.device)
        output = output.clone()
        output[..., indices_device] = 0
        return output

    handle = mlp.act.register_forward_hook(ablation_hook)

    return handle, indices, layer_idx

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [9]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/8192*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([ 7785,    15,   187,   187, 36062,    13,   253,  2846,   326,   556,
          644,  1925,   346,   783,  3228,   273, 44074,     3,   285,   346,
          783,  3228,   273, 10540,   995,   369, 11420,   275,  1249,   394,
         5331,   407,  4377,  7406, 26821,   313,   805,  1047,    14,   805,
         3547,   481,   733,  3395,   247,  2201,  4055,   323,  5454,   342])

Zeroed activations in 0.0% of units in layer 0
The capital of France is Paris.

Paris, the city that has been called "the City of Lights" and "the City of Love", was founded in 12th century by King Louis IX (1214-1272). It became a major center for trade with


$25\%$ of first layer

In [10]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/8192*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 7785,    15,   187,   187, 36062,    13,   253,  5347,  2846,   273,
         6181,    13,   369, 11420,   275,  1249,  3507,   407,  4377,  7406,
        26821,   347,   247,  4055,   323,  5454,   285, 19938,   342,  3060,
           15,   733,   556,   644,  1925,   346,   783,  3228,   273, 44074,
            3,  1580,   352,  3395,   271,  5213,  1313, 37489,  1309,   253])

Zeroed activations in 25.0% of units in layer 0
The capital of France is Paris.

Paris, the capital city of France, was founded in 1282 by King Louis IX as a center for trade and commerce with Europe. It has been called "the City of Lights" since it became an international metropolis during the


$50\%$ of first layer

In [11]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/8192*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253, 2846,   13,  285,  187,  783, 5347,   15,  380, 5347,  273, 6181,
         310,  247, 2586,  326,  556,  644,  323, 1580,  275,  253, 1533,   15,
         432,  697,  715,  281,  352,  715,  715,  715,  715,  715,  598,  715,
         715,  715,  715,  715,  715,  715,  715,  271,  715,  715,  715,  715,
         253,  715])

Zeroed activations in 50.0% of units in layer 0
The capital of France is the city, and
the capital. The capital of France is a country that has been for since in the world. from its into to it into into into into into up into into into into into into into into an into into into into the into


$75\%$ of first layer

In [12]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/8192*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([253, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187,
        187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187,
        187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187,
        187, 187, 187, 187, 187, 187, 187, 187])

Zeroed activations in 75.0% of units in layer 0
The capital of France is the



















































$100\%$ of first layer

In [13]:
handle, indices, layer_id = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

inputs = tokenizer("The capital of France is", return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {len(indices)/8192*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([187, 313,  15,  13, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187,
        187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187,
        187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187, 187,
        187, 187, 187, 187, 187, 187, 187, 187])

Zeroed activations in 100.0% of units in layer 0
The capital of France is
 (.,
















































**Attention Head Ablation Experiments**

First let's check the attention heads

In [15]:
print(model.gpt_neox.layers[0].attention)

GPTNeoXAttention(
  (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
  (dense): Linear(in_features=2048, out_features=2048, bias=True)
)


We see that the attention mechanism expects a 2048d input (the token embedding or output from the previous block), which gets turned into a 6144-d vector. This corresponds to calculating its query, key, and value vectors and concatenating these. There are 8 heads computing the attention mechanisms between QKV in slices of 356d vectors.

These attention contributions from each head are then concatenated to form another 2048d vector, which in turn gets transformed into the final attention contribution vector that combines and mixes the computations from all heads, and is incorporated into the input for the next transformer block.

In [16]:
print(model.gpt_neox.layers[0].attention.dense)

Linear(in_features=2048, out_features=2048, bias=True)


In [17]:
model.config.hidden_size

2048

In [18]:
model.config.num_attention_heads

8

In [19]:
model.config.hidden_size // model.config.num_attention_heads

256

Attention head ablation function

In [20]:
def add_head_ablation(model, head_indices, layer_idx):

    num_layers = model.config.num_hidden_layers
    #Embedding dimension
    hidden = model.config.hidden_size
    #Number of attention heads
    num_heads = model.config.num_attention_heads
    #Projection dimension per attention head
    head_dim = hidden // num_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    # Load final attention module
    attention_dense = model.gpt_neox.layers[layer_idx].attention.dense

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    # Note this is a pre-hook because we want to intervene components before the forward pass.
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [21]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([253, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380,
        380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380,
        380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380, 380,
        380, 380, 380, 380, 380, 380, 380, 380])

Zeroed head [0] in layer 0
The capital of France is the The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The


Ablate 2 attention heads in layer 0

---



In [22]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 253,  380,  380,  380,  380,  380,  380,  380,  380,  359,  403,  247,
         359,  403,  247,  359, 1472,  275,  271,  891, 1353,  387,  253, 1755,
         273,  436,   15,  187,  187, 1231, 1849,  644, 2444,  327,  776,  747,
        4422,  323,  670,  767, 3618, 1024,   13,  285,  352,  434, 4720, 4704,
         281,  564])

Zeroed head [0, 1] in layer 0
The capital of France is the The The The The The The The The we are a we are a we're in an i'm at the top of this.

We've been working on our new website for about two weeks now, and it's finally ready to go


Ablate 4 attention heads in layer 0

In [24]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([  253,   380,   844,   403,   247,   359,   452,   644,   275,  3874,
           15,   187,    29,  1288,  4725,  1231,  1472,   417,  1469,   281,
          320,  1060,   323,  1048,    13,   594,   309,  1353,   816,  6501,
          564,   896,  1728,   285,   755,   690,  4600, 17778,    81,    31,
          654,    66,  3860,   568,  2413,  1358,  2700,    19,    14,    18])

Zeroed head [0, 1, 2, 3] in layer 0
The capital of France is the The We are a we have been in town.
<br />We're not going to be here for long, so I'm just gonna go back home and get some sleep.<p> <a href="http://www2-1


Ablate 6 attention heads in layer 0

In [25]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0,1,2,3,4,5],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([  253,   380,  2846,   275,  7785,    13,   285,   359,   403,   247,
         1643,  6574,   432,   352,    15,   187,   187, 50274,    11, 50275,
           14, 50276,     3,    42,   717,   417,  2119,   604,   436,   588,
          789,   323,   368,   449,   428,   346,  1394,   476,   626,   320,
         2176,   670,   326,  1476,   535, 50270,  1042,  3003, 14732,  3481])

Zeroed head [0, 1, 2, 3, 4, 5] in layer 0
The capital of France is the The city in Paris, and we are a few miles from it.

    *   -  "I am not sure if this will work for you." - "You can't be certain about that!"

        IN THE COURT OF


Ablate 8 (all) attention heads in layer 0

In [26]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(8)],
    layer_idx=0,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([ 510, 5347,  273, 6181,  310])
Output tokens: tensor([  253,   806,  2846,   275,   247,  2586,   187,    29,  1288,    31,
        50276,  1171,  4343,    15,   380,   654,    66,  3860,   568,  2413,
         1358,  2700,    14,    67,   336,    78,  3024,    19,    69,    17,
           68,    18,   909,    73,    89,    72,    75,    90,    71,    76,
           91,    77,    88,  1138,  7053,   870,    74, 13208,   209,  2756])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
The capital of France is the first city in a country
<br>  of countries. The <a href="http://www-bcemnet2d0c1eghxgjyfkzlw">first</i>. 





Ablate all attention heads in layer 11

In [27]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(8)],
    layer_idx=11,
)

inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([ 510, 5347,  273, 6181,  310])
Outputs: tensor([ 7785,    15,   187,  1147,   434,   247,  2846,   326,   556,   644,
         1475,   323, 14020,    13,   285,   352,   369,   253,   806,   281,
          320,  4270,   407,  7497,   275,   752,   359,  1024,  1067,  3060,
          313,   395,   534,  3797,  4243,   751,  1457,  2816,   481,   733,
          671,  6569,   417,   760,   984,   627,   403,  1142,   952,   665])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 11
The capital of France is Paris.
It's a city that has been around for centuries, and it was the first to be built by humans in what we now call Europe (and which includes parts like New York). It also happens not only because there are many people who


Ablate heads in every layer

Baseline

In [28]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Baseline
The capital of France is Paris.

Paris, the city that has been called "the City of Lights" and "the City of Love", was founded in 12th century by King Louis IX (1214-1272). It became a major center for trade with


In [29]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed head 0 in all layers
The capital of France is not in the box.

I’m sorry, I know you are but it was a good idea to have an extra layer on top of your head when you get hit by lightning and that's why we're all here for this special event


In [32]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3 in all layers
The capital of France is the city, but it's not.
  (the town) and that means "but I'm aint got no money." The Town" (I'll be on my self.) . But you're in't!


In [33]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
ablations = { x: [0,1,2,3,4,5,6,7] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7 in all layers
The capital of France isometricsorry, and the same-a ndividualityou canvice.
============> The New York Times Square One World Warwicket alphabetically speaking to beacross from a) (originally published in this yearnts


In [34]:
inputs = tokenizer(
    '''The capital of France is''',
    return_tensors="pt")

num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads
ablations = { x: [y for y in range(num_heads)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([ 7480,    84,  5163,    13,   285,   253,  1072,    14,    66, 40515,
         1958,   414,   276,   476,    87,   547,    15,   187,  7770,    31,
          380,  1457,  2816,  7717, 15894,  2596,  3645,  3660, 15989,   292,
          355, 20376, 17859,  8288,   281,   320,   317,  1350,   432,   247,
           10,   313,   263, 20306,  3863,   275,   436,   807,    79,  1641])

Zeroed all heads in all layers
The capital of France isometricsorry, and the same-a ndividualityou canvice.
============> The New York Times Square One World Warwicket alphabetically speaking to beacross from a) (originally published in this yearnts


Ablating all heads seemed to make the model predict a bunch of newline characters unless I increased the reptition penalty.

In [35]:
print(repr(tokenizer.decode([187])))
print(tokenizer.convert_ids_to_tokens(187))

'\n'
Ċ
